**Estrutura Base e Ambiente**

In [11]:
import random
from dataclasses import dataclass
from enum import Enum

# Definição de tipos e posições
Position = tuple[int, int]

class ActionType(Enum):
    MOVE_UP = "MOVE_UP"
    MOVE_DOWN = "MOVE_DOWN"
    MOVE_LEFT = "MOVE_LEFT"
    MOVE_RIGHT = "MOVE_RIGHT"
    WAIT = "WAIT"

@dataclass(frozen=True)
class Observation:
    step: int
    self_position: Position
    visible_cells: dict[Position, str]

class WarehouseEnv:
    def __init__(self, width=10, height=5, num_packages=5):
        self.width = width
        self.height = height
        self.packages = set()
        # Posicionamento aleatório de pacotes (Recursos)
        while len(self.packages) < num_packages:
            self.packages.add((random.randint(0, width-1), random.randint(0, height-1)))

        self.agent_positions = {"Supervisor": (0, 0), "Operario": (width-1, height-1)}
        self.scores = {"Supervisor": 0, "Operario": 0}

    def get_observation(self, agent_name: str, step: int, radius=2) -> Observation:
        pos = self.agent_positions[agent_name]
        visible = {}
        # Implementação de sensores (Percepção Parcial)
        for y in range(pos[1]-radius, pos[1]+radius+1):
            for x in range(pos[0]-radius, pos[0]+radius+1):
                if 0 <= x < self.width and 0 <= y < self.height:
                    visible[(x, y)] = "package" if (x, y) in self.packages else "empty"
        return Observation(step, pos, visible)

    def apply_action(self, agent_name: str, action: ActionType):
        pos = self.agent_positions[agent_name]
        deltas = {
            ActionType.MOVE_UP: (0, -1),
            ActionType.MOVE_DOWN: (0, 1),
            ActionType.MOVE_LEFT: (-1, 0),
            ActionType.MOVE_RIGHT: (1, 0),
            ActionType.WAIT: (0, 0)
        }
        dx, dy = deltas[action]
        new_pos = (pos[0] + dx, pos[1] + dy)

        # Atuadores com checagem de limites
        if 0 <= new_pos[0] < self.width and 0 <= new_pos[1] < self.height:
            self.agent_positions[agent_name] = new_pos
            if new_pos in self.packages:
                self.packages.remove(new_pos)
                self.scores[agent_name] += 10 # Medida de Desempenho

    def render(self):
        grid = [["." for _ in range(self.width)] for _ in range(self.height)]
        for x, y in self.packages: grid[y][x] = "*"
        grid[self.agent_positions["Supervisor"][1]][self.agent_positions["Supervisor"][0]] = "S"
        grid[self.agent_positions["Operario"][1]][self.agent_positions["Operario"][0]] = "O"
        print("\n" + "+" + "-" * self.width + "+")
        for row in grid: print("|" + "".join(row) + "|")
        print("+" + "-" * self.width + "+")
        print(f"Scores: {self.scores}")

**Agente 1 - Supervisor (LLM)**

In [12]:
class ModelBasedWorkerAgent:
    def __init__(self):
        self.known_packages = set() # O "Modelo" interno do ambiente
        self.current_pos = (0, 0)

    def observe(self, obs: Observation):
        self.current_pos = obs.self_position
        # Atualiza a memória baseada no que está vendo agora
        for pos, cell_type in obs.visible_cells.items():
            if cell_type == "package":
                self.known_packages.add(pos)
            elif cell_type == "empty" and pos in self.known_packages:
                self.known_packages.discard(pos)

    def decide(self) -> ActionType:
        if not self.known_packages:
            return ActionType.WAIT

        # Persegue o objetivo baseado na memória (pacote mais próximo conhecido)
        target = min(self.known_packages, key=lambda p: abs(p[0]-self.current_pos[0]) + abs(p[1]-self.current_pos[1]))
        dx = target[0] - self.current_pos[0]
        dy = target[1] - self.current_pos[1]

        if abs(dx) >= abs(dy) and dx != 0:
            return ActionType.MOVE_RIGHT if dx > 0 else ActionType.MOVE_LEFT
        if dy != 0:
            return ActionType.MOVE_DOWN if dy > 0 else ActionType.MOVE_UP
        return ActionType.WAIT

**Agente 2 - Operário (Baseado em Modelo**

In [13]:
class ModelBasedWorkerAgent:
    def __init__(self):
        self.known_packages = set() # Memória interna do mundo
        self.current_pos = (0, 0)

    def observe(self, obs: Observation):
        self.current_pos = obs.self_position
        # Atualiza o modelo interno com novas informações visíveis
        for pos, type in obs.visible_cells.items():
            if type == "package":
                self.known_packages.add(pos)
            elif type == "empty" and pos in self.known_packages:
                self.known_packages.discard(pos)

    def decide(self) -> ActionType:
        if not self.known_packages:
            return ActionType.WAIT # Reflexo simples se não souber de nada [cite: 16]

        # Baseado em objetivos: encontrar o caminho para o pacote conhecido mais próximo [cite: 16]
        target = min(self.known_packages, key=lambda p: abs(p[0]-self.current_pos[0]) + abs(p[1]-self.current_pos[1]))
        dx = target[0] - self.current_pos[0]
        dy = target[1] - self.current_pos[1]

        if abs(dx) >= abs(dy) and dx != 0:
            return ActionType.MOVE_RIGHT if dx > 0 else ActionType.MOVE_LEFT
        if dy != 0:
            return ActionType.MOVE_DOWN if dy > 0 else ActionType.MOVE_UP
        return ActionType.WAIT

**Execução da Simulação**

In [14]:
# Inicialização
env = WarehouseEnv(width=10, height=5, num_packages=6)
agent_llm = LLMSupervisorAgent()
agent_model = ModelBasedWorkerAgent()

print("Simulação iniciada: Supervisor (S) e Operário (O)")

for step in range(20):
    # Turno do Supervisor
    obs_s = env.get_observation("Supervisor", step)
    action_s = agent_llm.decide(obs_s)
    env.apply_action("Supervisor", action_s)

    # Turno do Operário
    obs_o = env.get_observation("Operario", step)
    agent_model.observe(obs_o)
    action_o = agent_model.decide()
    env.apply_action("Operario", action_o)

    # Renderização
    print(f"\n--- PASSO {step:02d} ---")
    print(f"S-LLM: {action_s.value} | Motivo: {agent_llm.last_reason}")
    print(f"O-Modelo: {action_o.value} | Pacotes em memória: {len(agent_model.known_packages)}")
    env.render()

Simulação iniciada: Supervisor (S) e Operário (O)

--- PASSO 00 ---
S-LLM: MOVE_RIGHT | Motivo:  Indo coletar o pacote próximo. 
O-Modelo: WAIT | Pacotes em memória: 0

+----------+
|.S..*.....|
|.......*..|
|..........|
|.*....*...|
|....**...O|
+----------+
Scores: {'Supervisor': 0, 'Operario': 0}

--- PASSO 01 ---
S-LLM: MOVE_RIGHT | Motivo:  Indo coletar o pacote próximo. 
O-Modelo: WAIT | Pacotes em memória: 0

+----------+
|..S.*.....|
|.......*..|
|..........|
|.*....*...|
|....**...O|
+----------+
Scores: {'Supervisor': 0, 'Operario': 0}

--- PASSO 02 ---
S-LLM: MOVE_RIGHT | Motivo:  Indo coletar o pacote próximo. 
O-Modelo: WAIT | Pacotes em memória: 0

+----------+
|...S*.....|
|.......*..|
|..........|
|.*....*...|
|....**...O|
+----------+
Scores: {'Supervisor': 0, 'Operario': 0}

--- PASSO 03 ---
S-LLM: MOVE_RIGHT | Motivo:  Indo coletar o pacote próximo. 
O-Modelo: WAIT | Pacotes em memória: 0

+----------+
|....S.....|
|.......*..|
|..........|
|.*....*...|
|....**...O|
